# Bronze: metadata-driven API ingestion

**Layer:** Bronze (ELT: extract and load first, transform later)
**One notebook, many sources.** The pipeline (or you) passes `source_name`. Everything else comes from `api_sources.json`.

## What this notebook does, in order
1. Read the source's entry from the config file
2. Read the last watermark from `meta.watermark` (first run: use `watermark_default`)
3. Fetch pages from the API (retry, paging, pause between calls)
4. **Land the raw responses first**, exactly as received, in a batch folder
5. Read the batch folder, extract records, check the schema
6. Flatten to columns and add audit columns
7. Append to the bronze Delta table (a rerun of the same batch replaces that batch, never duplicates it)
8. Move the watermark **last**, only after the write succeeded
9. Write one row to `meta.run_log` whether the run succeeded or failed, and re-raise on failure so the pipeline shows red

## Rules this notebook follows
- Bronze is append-only history. No cleaning, no deduplication, no business rules (that is silver's job).
- Secrets never live in the config or the notebook. The config holds only a secret *name*; fetch the value from Key Vault.
- The watermark only moves forward after a successful write.
- Raw files are kept so any batch can be replayed without calling the API again (`replay_batch_id`).

## Before the first run
- Lakehouse attached as default, with schemas `bronze`, `silver`, `meta` enabled
- Config uploaded to `Files/config/api_sources.json`

## 1. Parameters
Mark the next cell as a **parameter cell** (cell menu, Toggle parameter cell) so a pipeline can override these values.

In [ ]:
source_name = "github_issues"        # which entry of the config to run
config_path = "/lakehouse/default/Files/config/api_sources.json"
replay_batch_id = ""                 # set to an existing batch id to re-parse its raw files without calling the API

## 2. Imports and constants

In [ ]:
import json
import os
import time
import uuid
import datetime as dt

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, ArrayType, MapType

try:                                            # pyspark 3.4+
    from pyspark.errors import AnalysisException
except ImportError:                             # older runtimes
    from pyspark.sql.utils import AnalysisException

# Store and compare all timestamps in UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

LAKEHOUSE_ROOT = "/lakehouse/default"           # how Python reaches the default lakehouse
ISO_FORMAT = "%Y-%m-%dT%H:%M:%SZ"               # the one watermark format used everywhere

# Keys every source entry must have. A missing key fails fast with a clear message.
REQUIRED_KEYS = [
    "source_name", "base_url", "endpoint", "pagination_type", "page_param",
    "page_size_param", "page_size", "max_pages", "load_type",
    "watermark_param", "watermark_field", "watermark_type", "watermark_default",
    "records_path", "expected_columns", "bronze_folder", "bronze_table",
]

## 3. Config and time helpers

In [ ]:
def load_source(path, name):
    """Return the config entry for `name`, or raise a clear error."""
    with open(path) as f:
        sources = json.load(f)
    src = next((s for s in sources if s.get("source_name") == name), None)
    if src is None:
        raise ValueError(f"source_name '{name}' not found in {path}")
    missing = [k for k in REQUIRED_KEYS if k not in src]
    if missing:
        raise ValueError(f"Config for '{name}' is missing keys: {missing}")
    return src


def parse_iso(text):
    """'2026-10-27T12:00:00Z' -> timezone-aware UTC datetime."""
    return dt.datetime.fromisoformat(text.replace("Z", "+00:00")).astimezone(dt.timezone.utc)


def to_iso(value, watermark_type):
    """Normalize a watermark value from the data to ISO text. Epoch milliseconds become a UTC date."""
    if value is None:
        return None
    if watermark_type == "epoch_ms":
        moment = dt.datetime.fromtimestamp(int(value) / 1000, dt.timezone.utc)
    else:
        moment = parse_iso(str(value))
    return moment.strftime(ISO_FORMAT)


def request_start(src, last_watermark):
    """Where this run starts asking from: saved watermark (or default), minus the overlap window.
    The overlap re-reads a few minutes on purpose so late-arriving records are not missed.
    Duplicates it causes are removed later by the silver merge."""
    if src["load_type"] != "incremental":
        return None                                       # full load: no filter
    base = last_watermark or src["watermark_default"]
    start = parse_iso(base) - dt.timedelta(minutes=src.get("overlap_minutes", 0))
    return start.strftime(ISO_FORMAT)

## 4. Control tables (watermark and run log)
`meta.watermark` holds one row per source: where the last successful run stopped. `meta.run_log` holds one row per run, success or failure.

In [ ]:
def ensure_control_tables():
    """Create the control tables on purpose, so the first scheduled run does not create them by accident."""
    spark.sql("CREATE SCHEMA IF NOT EXISTS meta")
    spark.sql("""
        CREATE TABLE IF NOT EXISTS meta.watermark (
            source_name      STRING,
            last_watermark   STRING,
            last_run_status  STRING,
            last_run_time    TIMESTAMP
        ) USING DELTA
    """)
    spark.sql("""
        CREATE TABLE IF NOT EXISTS meta.run_log (
            run_id           STRING,
            source_name      STRING,
            layer            STRING,
            batch_id         STRING,
            status           STRING,
            rows_read        BIGINT,
            rows_written     BIGINT,
            pages_fetched    INT,
            watermark_start  STRING,
            watermark_end    STRING,
            error_message    STRING,
            started_at       TIMESTAMP,
            finished_at      TIMESTAMP
        ) USING DELTA
    """)


def utc_now():
    """Naive UTC datetime, which Spark stores as a UTC timestamp."""
    return dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)


def get_watermark(name):
    """Last saved watermark for a source, or None if it has never run."""
    rows = (spark.table("meta.watermark")
                 .filter(F.col("source_name") == name)
                 .select("last_watermark").collect())
    return rows[0]["last_watermark"] if rows else None


def save_watermark(name, value, status):
    """Upsert the single row for this source. MERGE is atomic, unlike delete-then-insert."""
    new_row = spark.createDataFrame(
        [(name, value, status, utc_now())],
        "source_name string, last_watermark string, last_run_status string, last_run_time timestamp",
    )
    (DeltaTable.forName(spark, "meta.watermark").alias("t")
        .merge(new_row.alias("s"), "t.source_name = s.source_name")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())


def log_run(**f):
    """Append one audit row for this run."""
    row = (f["run_id"], f["source_name"], "bronze", f.get("batch_id"), f["status"],
           f.get("rows_read"), f.get("rows_written"), f.get("pages_fetched"),
           f.get("watermark_start"), f.get("watermark_end"), f.get("error_message"),
           f["started_at"], utc_now())
    schema = ("run_id string, source_name string, layer string, batch_id string, status string, "
              "rows_read bigint, rows_written bigint, pages_fetched int, watermark_start string, "
              "watermark_end string, error_message string, started_at timestamp, finished_at timestamp")
    spark.createDataFrame([row], schema).write.format("delta").mode("append").saveAsTable("meta.run_log")

## 5. Fetch: retry, paging, land raw
Two paging styles are supported, chosen by `pagination_type` in the config: `page` (page numbers, like GitHub) and `offset` (start position, like USGS). Retries cover 429 and 5xx errors and honor the `Retry-After` header. 4xx errors other than 429 are not retried, because repeating a bad request never fixes it.

In [ ]:
def build_session():
    """HTTP session with automatic retry and exponential backoff."""
    retry = Retry(
        total=5,
        backoff_factor=2,                        # waits 2s, 4s, 8s, ...
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        respect_retry_after_header=True,
    )
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.headers.update({"Accept": "application/json", "User-Agent": "fabric-ingestion"})
    # Authentication would go here. Fetch the secret from Key Vault using src["auth_secret_name"];
    # never print it, never write it to the config, never log request headers.
    return session


def extract_records(body, records_path):
    """Follow a dotted path ('features' or 'data.items') to the list of records. Empty path = body is the list."""
    node = body
    for part in filter(None, records_path.split(".")):
        node = node[part]
    return node


def fetch_and_land(src, session, start_value, raw_dir):
    """Fetch up to max_pages pages and save each response untouched.
    Returns (records_per_page, truncated). truncated=True means the page cap stopped the run
    while data may still be waiting."""
    url = src["base_url"] + src["endpoint"]
    params_base = {src["page_size_param"]: src["page_size"], **src.get("extra_params", {})}
    if start_value:
        params_base[src["watermark_param"]] = start_value

    pages, offset = [], src.get("offset_start", 1)
    for page_num in range(1, src["max_pages"] + 1):
        params = dict(params_base)
        if src["pagination_type"] == "page":
            params[src["page_param"]] = page_num
        elif src["pagination_type"] == "offset":
            params[src["page_param"]] = offset
        else:
            raise ValueError(f"Unsupported pagination_type: {src['pagination_type']}")

        resp = session.get(url, params=params, timeout=30)
        if resp.status_code == 204:              # some APIs answer 204 (no content) when nothing matches
            break
        resp.raise_for_status()                  # a 4xx here is a real error: fail loudly

        body = resp.json()
        records = extract_records(body, src["records_path"])
        if not records:
            break

        # Land the whole response exactly as received, before any parsing.
        with open(f"{raw_dir}/page_{page_num:04d}.json", "w") as f:
            json.dump(body, f)
        pages.append(len(records))

        if len(records) < src["page_size"]:      # a short page is the last page
            break
        offset += src["page_size"]
        time.sleep(src.get("sleep_seconds", 0))  # be polite to the API
    else:
        return pages, True                       # loop ran to max_pages with full pages
    return pages, False

## 6. Parse, schema check and flatten
The check compares the fields the config *expects* (`expected_columns`) with what arrived. A missing field stops the run. A new, unexpected top-level field is reported and ignored (add it to `expected_columns` when you want it). Dotted names such as `properties.mag` reach inside nested objects and become `properties_mag`. Lists and objects that remain are kept as JSON text.

In [ ]:
def read_raw_batch(batch_rel_dir, records_path):
    """Read every saved page of one batch. Returns one row per record plus the source file name."""
    raw = (spark.read.option("multiLine", True).json(f"{batch_rel_dir}/page_*.json")
                .withColumn("source_file", F.input_file_name()))   # capture it now, before any select drops it
    if records_path:                                              # records are wrapped inside a field
        raw = (raw.select(F.explode(F.col(records_path)).alias("rec"), "source_file")
                  .select("rec.*", "source_file"))
    return raw


def path_exists(df, path):
    """True if a (possibly dotted) column path exists in the data frame."""
    try:
        df.select(F.col(path))
        return True
    except AnalysisException:
        return False


def check_schema(df, expected_columns):
    """Fail on missing expected fields; report new fields. Returns the list of new top-level fields."""
    missing = [c for c in expected_columns if not path_exists(df, c)]
    if missing:
        raise ValueError(f"Schema drift: expected fields missing from the response: {missing}")
    known = {c.split(".")[0] for c in expected_columns} | {"source_file"}
    return sorted(set(df.columns) - known)


def flatten(df, expected_columns):
    """Select the expected fields with clean names. Nested objects/lists that remain become JSON text."""
    cols = []
    for path in expected_columns:
        col = F.col(path)
        data_type = df.select(col).schema[0].dataType
        if isinstance(data_type, (StructType, ArrayType, MapType)):
            col = F.to_json(col)
        cols.append(col.alias(path.replace(".", "_")))
    return df.select(*cols, "source_file")

## 7. Main run
The whole run sits in one function with one `try/except`. On any error it writes a failure row to the run log and re-raises, so the pipeline shows red. The watermark is touched only on the last success line.

In [ ]:
def run(source_name, replay_batch_id=""):
    run_id, started_at = str(uuid.uuid4()), utc_now()
    src = load_source(config_path, source_name)

    if not src.get("is_active", 1):
        print(f"{source_name} is switched off in the config (is_active = 0). Nothing to do.")
        return

    replay = bool(replay_batch_id)
    # Batch id = UTC time + short random suffix, so two runs in the same second can never collide.
    batch_id = replay_batch_id or f"{dt.datetime.now(dt.timezone.utc).strftime('%Y%m%d_%H%M%S')}_{run_id[:6]}"
    batch_rel_dir = f"{src['bronze_folder']}/batch_{batch_id}"     # path Spark uses
    raw_dir = f"{LAKEHOUSE_ROOT}/{batch_rel_dir}"                  # path Python uses
    incremental = src["load_type"] == "incremental"

    info = dict(run_id=run_id, source_name=source_name, batch_id=batch_id, started_at=started_at)
    try:
        ensure_control_tables()
        last_watermark = get_watermark(source_name)
        start_value = request_start(src, last_watermark)
        info["watermark_start"] = start_value
        print(f"[{source_name}] batch {batch_id}  replay={replay}  start from: {start_value}")

        # 1) Extract: call the API and land the raw pages (skipped on replay)
        pages, truncated = [], False
        if not replay:
            os.makedirs(raw_dir, exist_ok=True)
            pages, truncated = fetch_and_land(src, build_session(), start_value, raw_dir)
            info["pages_fetched"] = len(pages)
            print(f"[{source_name}] records per page: {pages}  total: {sum(pages)}")
            if not pages:
                log_run(status="success_no_data", **info)
                print(f"[{source_name}] nothing new. Watermark unchanged.")
                return

            # Safety rule. If the page cap stopped us and the API does not return records in
            # watermark order, moving the watermark could skip the records we did not fetch.
            if truncated and incremental and not src.get("sorted_by_watermark", False):
                raise RuntimeError(
                    "Hit max_pages and this source is not sorted by its watermark field. "
                    "Raise max_pages or narrow the window; the watermark was not moved.")

        # 2) Load: read the raw batch, check the schema, flatten, add audit columns
        raw_df = read_raw_batch(batch_rel_dir, src["records_path"])
        rows_read = raw_df.count()
        extra_fields = check_schema(raw_df, src["expected_columns"])
        if extra_fields:
            print(f"[{source_name}] NEW fields in the response (ignored): {extra_fields}")

        new_watermark = None
        if incremental:                                   # compute from the raw data, before any select
            top = raw_df.agg(F.max(F.col(src["watermark_field"]))).collect()[0][0]
            new_watermark = to_iso(top, src["watermark_type"])

        bronze_df = (flatten(raw_df, src["expected_columns"])
                     .withColumn("ingest_ts", F.current_timestamp())
                     .withColumn("source_name", F.lit(source_name))
                     .withColumn("batch_id", F.lit(batch_id)))

        # Idempotent write: a replay of the same batch replaces that batch instead of duplicating it.
        if spark.catalog.tableExists(src["bronze_table"]):
            spark.sql(f"DELETE FROM {src['bronze_table']} WHERE batch_id = '{batch_id}'")
        bronze_df.write.format("delta").mode("append").saveAsTable(src["bronze_table"])

        # 3) Move the watermark LAST, only after the write succeeded
        if incremental and new_watermark and not replay:
            save_watermark(source_name, new_watermark, "success")
        info["watermark_end"] = new_watermark or last_watermark

        log_run(status="success", rows_read=rows_read, rows_written=rows_read, **info)
        print(f"[{source_name}] done. rows written: {rows_read}  new watermark: {info['watermark_end']}")

    except Exception as err:
        try:
            log_run(status="failed", error_message=str(err)[:2000], **info)
        except Exception as log_err:                       # never let logging hide the real error
            print(f"Could not write the failure row to meta.run_log: {log_err}")
        raise                                              # always re-raise so the pipeline run fails

In [ ]:
run(source_name, replay_batch_id)

## 8. Checks to run by hand after a load
```sql
-- last runs for this source
SELECT * FROM meta.run_log WHERE source_name = 'github_issues' ORDER BY started_at DESC LIMIT 5;

-- where the next run will start
SELECT * FROM meta.watermark;

-- rows per batch in bronze
SELECT batch_id, COUNT(*) FROM bronze.github_issues GROUP BY batch_id ORDER BY batch_id;
```

## Notes for production
- **Orchestration:** run this notebook from a pipeline (Lookup over the config, ForEach, Notebook activity with `source_name` as a base parameter). The pipeline gives you scheduling, retries per source, alerts and run history.
- **Shared code:** the helper functions would live in a shared module or environment library once there is more than one ingestion notebook, so there is one copy to fix.
- **Ask first at the company:** which config, control-table, logging and secret-store conventions already exist. Use those instead of these.
- **Data rules:** do not put real credentials or restricted data into outside tools or a public repo.